In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

from dataclasses import dataclass
import znnl as nl
from neural_tangents import stax
import optax
from copy import deepcopy
import matplotlib.pyplot as plt
import jax
import numpy as np
import jax.numpy as jnp
import flax.linen as nn
from znnl.optimizers import TraceOptimizer
from znnl.utils.prng import PRNGKey


In [ ]:
for i in range(10):
        
    class CustomModule(nn.Module):
        """
        Simple CNN module.
        """
        @nn.compact
        def __call__(self, x):
            x = x.reshape((x.shape[0], -1))  # flatten
            x = nn.Dense(features=128)(x)
            x = nn.relu(x)
            x = nn.Dense(features=128)(x)
            x = nn.relu(x)
            x = nn.Dense(features=10)(x)
            return x

    mnist_model = nl.models.FlaxModel(
            flax_module=CustomModule(),
            optimizer=optax.sgd(learning_rate=1e-4),
            input_shape=(1, 28, 28, 1),
            batch_size=1
        )

    # get devices
    num_devices = len(jax.devices())

    #select dataset
    data_generator = nl.data.MNISTGenerator(ds_size = 1000)

    ds_agent = nl.agents.RandomAgent(
        data_generator=data_generator
    )
    _ = ds_agent.build_dataset(500)
    indices = ds_agent.target_indices

    train_ds = {
        "inputs": np.take(data_generator.train_ds["inputs"], indices, axis=0),
        "targets": np.take(data_generator.train_ds["targets"], indices, axis=0)
    }
    
     # Set the optimizer
    key = PRNGKey()
    # Select a subset of the data-set.
    indices = jax.random.randint(key(), shape=(20,), minval=0, maxval=train_ds["inputs"].shape[0])
    data_set = np.take(train_ds["inputs"], indices, axis=0)
    eta_start = np.trace(mnist_model.compute_ntk(data_set)["empirical"])
    
    optimizer = TraceOptimizer(
        scale_factor=eta_start, subset_size = 50, regularizer = True, delta_s = False
    )
    mnist_model.optimizer = optimizer

    #defining recorders 
    train_recorder = nl.training_recording.JaxRecorder(
        name=f"train_recorder_{i}",
        loss=True,
        accuracy=True,
        update_rate=1,
        chunk_size=100,
        storage_path="TraceOpt-Reg1/"
    )
    train_recorder.instantiate_recorder(
        data_set=train_ds
    )

    test_recorder = nl.training_recording.JaxRecorder(
        name=f"test_recorder_{i}",
        loss=True,
        accuracy=True,
        update_rate=1, 
        chunk_size=100,
        storage_path="TraceOpt-Reg1/"
    )
    test_recorder.instantiate_recorder(
        data_set=data_generator.test_ds
    )

    #defining training strategy
    training_strategy = nl.training_strategies.SimpleTraining(
        model=mnist_model, 
        loss_fn=nl.loss_functions.CrossEntropyLoss(),
        accuracy_fn=nl.accuracy_functions.LabelAccuracy(),
        recorders=[train_recorder, test_recorder],
    )

    #running the model for the wanted epochs
    batched_training_metrics = training_strategy.train_model(
        train_ds=train_ds,
        test_ds=data_generator.test_ds,
        batch_size=64,
        epochs=5000,
    )

    #Dump the records 
    train_recorder.dump_records()
    test_recorder.dump_records()